# 06 — Models (multi-city, config-selectable)

Architecture definitions only -- no training loop, no normalization
(both are `07`'s job). Builds every scenario (A-F; G separate, no torch
model) and runs one real forward pass per scenario against an actual
graph pair from `05`'s combined, multi-city `dataset_index.parquet`, to
catch shape/dimension bugs before a full `07` run.

**Absorbs the old `06b`'s job too** -- `MODEL_CONFIG_NAME` below picks
which `configs/model_*.yaml` to QC (`model.yaml` by default; set it to
`model_bootstrap.yaml`, `model_capacity_revision.yaml`,
`model_gin_comparison.yaml`, etc. to QC any other branch's capacity
instead). `build_model` already takes every capacity knob as a plain
argument, so swapping configs here is just a different set of arguments,
never a code change -- no reason to keep a separate `06X` notebook per
config the way `06b` used to be.

**Vocab sizes are read dynamically from the post-`04b` unified vocab
cache**, never hardcoded -- the old `06`/`06b` hardcoded
`BUILDING_TYPE_VOCAB = 58` / `HIGHWAY_VOCAB = 13` (correct only for
Bogor alone), which is exactly the kind of stale hardcoded vocab size
that earlier caused a CUDA `device-side assert` (out-of-bounds
embedding index) once a second city was pooled in. With N cities this
would just be wrong by a larger margin, not more forgiving -- so this
notebook fails loudly with an `assert` instead if the sizes look
Bogor-only-sized.

**Scenario F is included** (previously deferred in `06`/`06b`) -- it's
a real, non-placeholder scenario now (`models.UnifiedEncoder`, see
`src/unified_graph.py`), so it gets the same forward-pass QC as A-E.

Uses `src/models.py`, `src/unified_graph.py`, `src/baseline_features.py`.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric pyyaml pandas

In [ ]:
# ── Pick which model config to QC -- replaces the old separate 06b notebook.
MODEL_CONFIG_NAME = "model.yaml"  # or "model_bootstrap.yaml", "model_capacity_revision.yaml", etc.

import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/{MODEL_CONFIG_NAME}") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
PROCESSED_DIR = Path(paths_cfg["processed_dir"])
INTERIM_DIR = Path(paths_cfg["interim_dir"])
SVG_DIR = PROCESSED_DIR / "svg_graphs"
TVG_DIR = PROCESSED_DIR / "tvg_graphs"

# Unified vocab sizes, read from the post-04b cache -- NEVER hardcoded
# (see notebook intro for why). Any city's cache holds the same unified
# vocab post-04b (04b's swap-in cell writes it to every city), so the
# first city in CITIES is as good as any other to read from.
import json
_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB = len(json.load(f))

# Confirm every OTHER city's cache agrees -- catches a partially-run 04b
# (unified on some cities, not others) before it silently misaligns the
# shared embedding table.
for city in CITIES[1:]:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    with open(cache_dir / "highway_vocab.json") as f:
        hw_n = len(json.load(f))
    with open(cache_dir / "building_type_vocab.json") as f:
        bt_n = len(json.load(f))
    assert hw_n == HIGHWAY_VOCAB and bt_n == BUILDING_TYPE_VOCAB, (
        f"{city}'s vocab cache ({hw_n} highway / {bt_n} building_type) disagrees with "
        f"{CITIES[0]}'s ({HIGHWAY_VOCAB} / {BUILDING_TYPE_VOCAB}) -- run 04b_vocab_unification "
        f"for every city before trusting this notebook.")

# SVG object vocabs are fixed, city-independent segmentation classes
# (src/svg_builder.py's CLASS_IDX) -- never per-city, never unified.
SIGNAGE_VOCAB = 5
LIGHT_POLE_VOCAB = 4
ROAD_MARKING_VOCAB = 2

print(f"Model config: {MODEL_CONFIG_NAME}")
print(f"Unified vocab (post-04b, all {len(CITIES)} cities agree): "
      f"highway={HIGHWAY_VOCAB}, building_type={BUILDING_TYPE_VOCAB}")

In [ ]:
import models
import torch

FUSION_DIM = model_cfg.get("fusion_dim") or model_cfg.get("hidden_dim", 64)
HEAD_HIDDEN = model_cfg.get("head_hidden", 32)
HEAD_DROPOUT = model_cfg.get("head_dropout", 0.35)
CONV_TYPE = model_cfg.get("conv_type", "gatv2")

svg_kwargs = dict(
    hidden_dim=model_cfg.get("hidden_dim", 64), heads=model_cfg.get("heads", 4),
    num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.35),
    signage_vocab=SIGNAGE_VOCAB, light_pole_vocab=LIGHT_POLE_VOCAB,
    road_marking_vocab=ROAD_MARKING_VOCAB, cat_embed_dim=model_cfg.get("cat_embed_dim", 2),
    conv_type=CONV_TYPE,
)
tvg_kwargs = dict(
    hidden_dim=model_cfg.get("hidden_dim", 64), heads=model_cfg.get("heads", 4),
    num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.35),
    building_type_vocab=BUILDING_TYPE_VOCAB, highway_vocab=HIGHWAY_VOCAB,
    building_type_embed_dim=model_cfg.get("building_type_embed_dim", 8),
    highway_embed_dim=model_cfg.get("highway_embed_dim", 4),
    conv_type=CONV_TYPE,
)
print(f"fusion_dim={FUSION_DIM}  head_hidden={HEAD_HIDDEN}  head_dropout={HEAD_DROPOUT}  conv_type={CONV_TYPE}")
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)

In [ ]:
# ── Build every scenario (A-F; G is separate, no torch model) ──
SCENARIOS = ["A", "B", "C", "D", "E", "F"]
HEAD_DEPTHS = ["linear", "mlp2"]  # both compared, per the earlier decision

built_models = {}
for scenario in SCENARIOS:
    for depth in HEAD_DEPTHS:
        key = f"{scenario}_{depth}"
        built_models[key] = models.build_model(
            scenario, fusion_dim=FUSION_DIM, head_depth=depth,
            head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
            use_ablation=False, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs,
        )
        n_params = sum(p.numel() for p in built_models[key].parameters())
        print(f"{key:10s} — {n_params:,} parameters")

In [ ]:
# ── QC: one real forward pass per scenario against an actual graph pair ──
# Catches shape/dimension bugs now, not 20 minutes into a 07 training run.
import unified_graph as ug

index_df = None
try:
    import pandas as pd
    index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
except FileNotFoundError:
    print("dataset_index.parquet not found yet — run 05 first for this QC cell to work.")

if index_df is not None:
    sample_pid = index_df["point_id"].iloc[0]
    svg_sample = torch.load(SVG_DIR / f"{sample_pid}.pt", weights_only=False)
    tvg_sample = torch.load(TVG_DIR / f"{sample_pid}.pt", weights_only=False)
    merged_sample = ug.merge_svg_tvg(svg_sample, tvg_sample)
    print(f"Testing against: {sample_pid}")

    for scenario in SCENARIOS:
        for depth in HEAD_DEPTHS:
            key = f"{scenario}_{depth}"
            model = built_models[key]
            model.eval()
            with torch.no_grad():
                try:
                    if scenario == "A":
                        out = model(svg_sample)
                    elif scenario == "B":
                        out = model(tvg_sample)
                    elif scenario == "F":
                        out = model(merged_sample)
                    else:
                        out = model(svg_sample, tvg_sample)
                    print(f"  ✅ {key}: output shape {tuple(out.shape)}, value {out.item():.4f}")
                except Exception as e:
                    print(f"  ❌ {key}: {type(e).__name__}: {e}")

In [ ]:
# ── Scenario G: build the tabular feature table (separate path, no torch) ──
import baseline_features

if index_df is not None:
    sample_ids = index_df["point_id"].tolist()[:5]  # small sample for this QC check
    feat_table = baseline_features.build_feature_table(sample_ids, SVG_DIR, TVG_DIR, torch)
    display(feat_table)

In [ ]:
print(f"Model config QC'd: {MODEL_CONFIG_NAME}")
print(f"Capacity: fusion_dim={FUSION_DIM}, head_hidden={HEAD_HIDDEN}, "
      f"head_dropout={HEAD_DROPOUT}, conv_type={CONV_TYPE}")
print()
print("Still open for 07: batch size, epoch cap, patience, split scheme,")
print("and per-fold/per-repeat normalization fitting.")
print()
print("Next: 07_train_eval.ipynb (or whichever 07x branch matches this config)")